In [ ]:
import sys
from pathlib import Path
from PIL import Image
from typing import Any

from tqdm import tqdm
import torch
import torch.nn as nn
from torch.types import Tensor
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as ts
from torchvision.transforms import Compose

import spark as sp
from spark.handle import gather_data_filepaths, gather_data_labels
from spark.processing import center_tensor, normalise

In [ ]:
def get_size(obj: object, default: Any = -1) -> int:
    """Computes how much memory storage is being used by input obj in [bytes]."""
    return sys.getsizeof(obj, default=default)

In [ ]:
class ImageDataset(Dataset):
    """
    Baseline AutoEncoder dataset configurator.
    """
    def __init__(
        self, data_path: str | Path,
        class_lbls: dict[int, str],
        transform: Compose | None,
    ) -> None:
        files_list = gather_data_filepaths(data_path, shuffle=True)
        self.data = sp.process_data(files_list, lambda img: Image.open(img).convert('L'), transform)
        self.labels = torch.tensor(gather_data_labels(files_list, class_lbls))
    
    def __len__(self) -> int:
        return len(self.data)
    
    def __getitem__(self, index) -> tuple[Tensor, Tensor]:
        sample = self.data[index]
        label = self.labels[index]
        return sample, label

In [ ]:
#BASEPATH: str = '/home/edoardo/Desktop/MockDataForDMs'
BASEPATH: str = '/mnt/d/MockDataForDMs'
DATASET_ID: str = 'mockPolyImgsDataset'
BATCH_SIZE: int = 250
VALID_SIZE: float = 0.2
PREPROCESSING: Compose = ts.Compose(
    [
        ts.PILToTensor(),
        center_tensor,
        normalise,
    ]
)
CLASS_NAMES: dict[int, str] = {
    0: 'circle',
    1: 'triangle',
    2: 'square',
    3: 'hexagon',
}

try:
    dataset: Dataset = sp.load_dataset(f'{BASEPATH}/{DATASET_ID}.pt')
except FileNotFoundError:
    print('Dataset not found, generating...')
    dataset: ImageDataset = ImageDataset(f'{BASEPATH}/ImgsMockDatasetDMs', CLASS_NAMES, PREPROCESSING)
    sp.save_dataset(dataset, f'{BASEPATH}/{DATASET_ID}.pt')

train_dl, valid_dl = sp.get_dataloaders(dataset, BATCH_SIZE, VALID_SIZE)

In [ ]:
from BASE_mockModels import MockAutoEncoder

IN_CHANNELS: int = 1
LATENT_DIM: int = 8

autoencoder = MockAutoEncoder(IN_CHANNELS, LATENT_DIM)
print(autoencoder)

In [ ]:
from itertools import islice
from typing import NamedTuple
import warnings

import torch.optim as opt
from torch.amp import GradScaler, autocast

from spark.inspect import OutputManager


class LatentVector(NamedTuple):
    """Latent vector container."""
    vals: Tensor
    lbls: Tensor

class TrainResults(NamedTuple):
    """
    Model trainer results. Contains the avg train and valid loss
    values and the latent space vectors with respective labels.
    NOTE:
        * the tensors in `latent_dmap` are NOT attached
          to the model Op. Graph (see OutputManager)
    """
    train_loss: list[float]
    valid_loss: list[float]
    latent_dmap: dict[int, LatentVector]


def train_model(
    params: dict[str, Any],
    epochs: int,
    learning_rate: float,
    train_dl: DataLoader,
    valid_dl: DataLoader,
    latent_space_hook: OutputManager,
) -> TrainResults:
    """
    Basic train routine.
    """
    def check_loss_val(loss_val: float, msg: str) -> bool:
        """Checks the current loss value and raises a warning if is NaN."""
        if not torch.isnan(loss_val):
            return True
        warnings.warn(msg)
        return False
    
    # config procedure/loss container
    avg_train_loss, avg_valid_loss = [], []
    latent_container: dict[int, LatentVector] = {}
    tdl_len, vdl_len = map(len, (train_dl, valid_dl))

    # setup model/optimiser/scaler for memory saving
    device = params['device']
    device_type = device.type if isinstance(device, torch.device) else 'cuda'
    model = params['model'].to(device)
    loss_fn = params['loss']
    optimiser = params['optimiser'](model.parameters(), lr=learning_rate)
    scheduler = params['scheduler'](optimiser, patience=5, factor=0.5)
    scaler = GradScaler(device)

    loop = tqdm(range(epochs))
    for epoch in loop:
        
        # ---------   TRAINING   ---------
        loop.set_description('Training Model')
        model.train()
        running_batches = 0
        running_train_loss = 0.0

        for batch, (x_batch, _) in enumerate(islice(train_dl, tdl_len)):   # NOTE: ignoring labels for the moment
            loop.set_postfix({'batch': f'{batch + 1}/{tdl_len}'})

            # optimisation step
            x_batch = x_batch.to(device)

            optimiser.zero_grad()
            with autocast(device_type=device_type):
                out = model(x_batch)
                loss_val = loss_fn(out, x_batch)  # NOTE: for autoencoders, the target is the input
            
            if check_loss_val(
                loss_val, f'Train loss NaN @ E: {epoch + 1} B: {batch + 1}',
            ):
                running_batches += 1
                running_train_loss += loss_val.item()

                scaler.scale(loss_val).backward()
                scaler.step(optimiser)
                scaler.update()
            
        # loss logging
        avg_train_loss.append(running_train_loss / max(running_batches, 1))
        

        # ---------  VALIDATION  ---------
        loop.set_description('Validating Model')
        model.eval()
        running_valid_batches = 0
        running_valid_loss = 0.0

        with torch.no_grad(), sp.forward_data_capture(model.encoder, latent_space_hook):
            for batch, (x_batch, y_batch) in enumerate(islice(valid_dl, vdl_len)):
                loop.set_postfix({'batch': f'{batch + 1}/{vdl_len}'})

                # validation step
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)

                with autocast(device_type=device_type):
                    v_out = model(x_batch)
                    v_loss_val = loss_fn(v_out, x_batch)
                
                # add labels to hook storage
                latent_space_hook.add_labels(y_batch)
                
                if check_loss_val(
                    v_loss_val, f'Valid loss NaN @ E: {epoch + 1}',
                ):
                    running_valid_batches += 1
                    running_valid_loss += v_loss_val.item()
            
        # loss logging and lr update
        avg_v_loss_val = running_valid_loss / max(running_valid_batches, 1)
        avg_valid_loss.append(avg_v_loss_val)
        scheduler.step(avg_v_loss_val)
        # log latent space vectors (merge batches + clear storage)
        latent_container[epoch] = LatentVector(*latent_space_hook.merge())
        latent_space_hook.clear()        
    
    results = TrainResults(avg_train_loss, avg_valid_loss, latent_container)
    return results

In [ ]:
from BASE_train_routine import training_setup

EPOCHS: int = 5
LR: int = 1e-2
DEVICE: int = 'cuda' if torch.cuda.is_available() else 'cpu'
LOSS_FN = nn.MSELoss()

params = training_setup(
    autoencoder, LOSS_FN, opt.Adam, opt.lr_scheduler.ReduceLROnPlateau, DEVICE,
)
latent_space_hook = OutputManager()

results: TrainResults = train_model(
    params=params,
    epochs=EPOCHS,
    learning_rate=LR,
    train_dl=train_dl,
    valid_dl=valid_dl,
    latent_space_hook=latent_space_hook,
)

In [ ]:
latent_data_path = f'{BASEPATH}/latentSpaceContainer.pt'
latent_container = results.latent_dmap

if not Path(latent_data_path).exists():
    torch.save(latent_container, f'{BASEPATH}/latentSpaceContainer.pt')
    print('Latent space vectors saved!')
else:
    print('Latent space vectors already saved!')